# NimbusImage Python API — Getting Started

This notebook walks through the major features of the `nimbusimage` Python API. It connects to a running NimbusImage backend and demonstrates:

1. **Connecting** to the server
2. **Browsing** datasets and metadata
3. **Fetching images** (single frames, stacks, composites)
4. **Working with annotations** (create, list, filter, geometry)
5. **Running workers** (annotation and property workers)
6. **Connections** between annotations
7. **Export** (JSON and CSV)
8. **History** (undo/redo)
9. **Sharing** and access control

## Prerequisites

```bash
cd nimbusimage
source .venv/bin/activate
pip install -e ".[dev]"
pip install matplotlib   # for visualization in this notebook
```

The backend must be running (`docker compose up -d` from the repo root).

## 1. Connect to the server

In [ ]:
import nimbusimage as ni

# Connect with explicit credentials
client = ni.connect("http://localhost:8080/api/v1", username="admin", password="password")

# Alternative: connect with a token
# client = ni.connect("http://localhost:8080/api/v1", token="your-token")

# Alternative: connect via environment variables (NI_API_URL, NI_TOKEN)
# client = ni.connect()

print(f"Connected as user: {client.user_id}")
print(f"API URL: {client.api_url}")

## 2. Browse datasets

In [ ]:
# List all accessible datasets
datasets = client.list_datasets()
print(f"Found {len(datasets)} datasets\n")

for d in datasets[:5]:
    print(f"  {d['_id']}  {d['name']}")

In [ ]:
# Open a dataset by ID (change this to one of your dataset IDs)
DATASET_ID = datasets[0]["_id"]
ds = client.dataset(DATASET_ID)

# Explore metadata (fetched lazily on first access)
print(f"Name:       {ds.name}")
print(f"Shape:      {ds.shape} (height, width)")
print(f"Channels:   {ds.channels}")
print(f"Num Z:      {ds.num_z}")
print(f"Num Time:   {ds.num_time}")
print(f"Num XY:     {ds.num_xy}")
print(f"Pixel size: {ds.pixel_size.to('um').value:.3f} um")
print(f"Dtype:      {ds.dtype}")
print(f"Frames:     {len(ds.frames)}")

## 3. Fetch images

Images are returned as numpy arrays. All coordinates default to 0 if not specified.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Single frame
img = ds.images.get(channel=0, z=0, time=0)
print(f"Single frame: {img.shape}, dtype={img.dtype}")

# Display
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img, cmap="gray")
axes[0].set_title(f"{ds.channels[0]} (channel 0)")

# Another channel
if ds.num_channels > 1:
    img2 = ds.images.get(channel=1, z=0, time=0)
    axes[1].imshow(img2, cmap="gray")
    axes[1].set_title(f"{ds.channels[1]} (channel 1)")

# Composite RGB using the server's layer settings (colors, contrast)
rgb = ds.images.get_composite(z=0, time=0, dtype="uint8")
axes[2].imshow(rgb)
axes[2].set_title("Composite (from layer settings)")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Z-stack: returns a 3D array (Z, H, W)
stack = ds.images.get_stack(channel=0, axis="z")
print(f"Z-stack: {stack.shape}")

# All channels at one location
all_channels = ds.images.get_all_channels(z=0, time=0)
print(f"All channels: {len(all_channels)} images of shape {all_channels[0].shape}")

# Crop a region
crop = ds.images.get(channel=0, z=0, crop=(200, 200, 400, 400))
print(f"Cropped: {crop.shape}")

## 4. Annotations

Annotations are the core data objects — points, polygons, and lines drawn on images.

In [ ]:
# List and count annotations
all_anns = ds.annotations.list()
print(f"Total annotations: {len(all_anns)}")

# Filter by shape
polygons = ds.annotations.list(shape="polygon")
points = ds.annotations.list(shape="point")
print(f"Polygons: {len(polygons)}, Points: {len(points)}")

# Filter by tags
# tagged = ds.annotations.list(tags=["nucleus"])
# print(f"Tagged 'nucleus': {len(tagged)}")

# Count (faster than list if you just need the number)
print(f"Polygon count: {ds.annotations.count(shape='polygon')}")

In [ ]:
# Create annotations
# Point annotation
point = ni.Annotation.from_point(
    x=500.0, y=300.0, channel=0,
    tags=["notebook-demo"],
    dataset_id=ds.id,
    location=ni.Location(xy=0, z=0, time=0),
)
created_point = ds.annotations.create(point)
print(f"Created point: {created_point.id}")

# Polygon annotation (manual coordinates)
polygon = ni.Annotation(
    id=None, shape="polygon",
    tags=["notebook-demo"],
    channel=0,
    location=ni.Location(xy=0, z=0, time=0),
    coordinates=[
        {"x": 100.0, "y": 100.0},
        {"x": 150.0, "y": 100.0},
        {"x": 150.0, "y": 150.0},
        {"x": 100.0, "y": 150.0},
    ],
    dataset_id=ds.id,
)
created_polygon = ds.annotations.create(polygon)
print(f"Created polygon: {created_polygon.id}")

# Bulk create
bulk = [
    ni.Annotation.from_point(
        x=float(200 + i * 20), y=200.0, channel=0,
        tags=["notebook-demo", "bulk"],
        dataset_id=ds.id,
    )
    for i in range(5)
]
created_bulk = ds.annotations.create_many(bulk)
print(f"Bulk created: {len(created_bulk)} annotations")

### Geometry helpers

Polygon annotations have geometry methods for converting to shapely objects and numpy masks.

In [ ]:
# Geometry methods on polygon annotations
ann = created_polygon

# Convert to shapely
poly = ann.polygon()
print(f"Shapely polygon: area={poly.area:.1f}, perimeter={poly.length:.1f}")

# Centroid
cx, cy = ann.centroid()
print(f"Centroid: ({cx:.1f}, {cy:.1f})")

# Boolean mask
mask = ann.get_mask(ds.shape)
print(f"Mask: {mask.shape}, {mask.sum()} pixels")

# Pixel coordinates
rows, cols = ann.get_pixels(ds.shape)
print(f"Pixels: {len(rows)} points")

# Create annotation from a shapely polygon
from shapely.geometry import Polygon
new_poly = Polygon([(300, 300), (360, 300), (360, 360), (300, 360)])
from_shapely = ni.Annotation.from_polygon(
    new_poly, channel=0, tags=["notebook-demo", "from-shapely"], dataset_id=ds.id
)
print(f"From shapely: {len(from_shapely.coordinates)} vertices")

# Create annotation from a numpy mask
test_mask = np.zeros((100, 100), dtype=bool)
test_mask[20:50, 30:70] = True
from_mask = ni.Annotation.from_mask(
    test_mask, channel=0, tags=["notebook-demo", "from-mask"], dataset_id=ds.id
)
print(f"From mask: {len(from_mask.coordinates)} vertices")

### Client-side filtering

Filter annotations without making additional API calls.

In [ ]:
# Filter by tags (client-side)
demo_anns = ni.filter_by_tags(all_anns, ["notebook-demo"])
print(f"Filtered by tag 'notebook-demo': {len(demo_anns)}")

# Filter by location
at_z0 = ni.filter_by_location(all_anns, z=0)
print(f"At z=0: {len(at_z0)}")

# Group by location
groups = ni.group_by_location(all_anns)
for (time, z, xy), group in list(groups.items())[:3]:
    print(f"  (time={time}, z={z}, xy={xy}): {len(group)} annotations")

### Visualize annotations on an image

In [ ]:
# Overlay annotations on the composite image
rgb = ds.images.get_composite(z=0, time=0, dtype="uint8")
local_anns = ni.filter_by_location(ds.annotations.list(), z=0)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(rgb)

for ann in local_anns:
    if ann.shape == "polygon" and len(ann.coordinates) >= 3:
        pts = [(c["x"], c["y"]) for c in ann.coordinates]
        pts.append(pts[0])  # close the polygon
        xs, ys = zip(*pts)
        ax.plot(xs, ys, "w-", linewidth=1, alpha=0.7)
    elif ann.shape == "point" and ann.coordinates:
        x, y = ann.coordinates[0]["x"], ann.coordinates[0]["y"]
        ax.plot(x, y, "y+", markersize=6)

ax.set_title(f"{len(local_anns)} annotations at z=0")
ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Run workers

Workers are Docker containers that perform computations on datasets. Use `client.list_workers()` to discover what's available, then `ds.annotations.compute()` or `ds.properties.compute()` to run them.

In [ ]:
# Discover available workers
workers = client.list_workers()
print(f"Available workers: {len(workers)}\n")

# Show annotation workers
for image, labels in sorted(workers.items()):
    if image.startswith("annotations/") and ":test" not in image:
        name = labels.get("interfaceName", "")
        print(f"  {image}  ({name})")

In [ ]:
# Inspect a worker's parameter interface
iface = client.get_worker_interface("annotations/random_squares:latest")
print("random_squares parameters:\n")
for name, spec in sorted(iface.items(), key=lambda x: x[1].get("displayOrder", 99)):
    if spec["type"] == "notes":
        continue
    default = spec.get("default", "")
    print(f"  {name}: type={spec['type']}, default={default}")

In [ ]:
# Run an annotation worker
count_before = ds.annotations.count()

job = ds.annotations.compute(
    image="annotations/random_squares:latest",
    channel=0,
    tags=["notebook-demo", "worker-squares"],
    worker_interface={
        "Number of squares": 10,
        "Square size": 15,
    },
    name="notebook_demo",
)

print(f"Job submitted: {job.id}")
success = job.wait(poll_interval=2.0, verbose=True)
print(f"\nSuccess: {success}")
print(f"Annotations: {count_before} -> {ds.annotations.count()}")

In [ ]:
# Run a property worker
prop = ds.properties.get_or_create(
    "Notebook Blob Metrics", shape="polygon",
    image="properties/blob_metrics:latest",
)
ds.properties.register(prop.id)
print(f"Property: {prop.name} ({prop.id})")

# Get scales from collection settings
scales = ds.collections.get_raw().get("meta", {}).get("scales", {})

job = ds.properties.compute(
    prop,
    worker_interface={"Use physical units": False},
    scales=scales,
)

print(f"Job submitted: {job.id}")
success = job.wait(poll_interval=2.0, verbose=False)
print(f"Success: {success}")

# Check computed values
values = ds.properties.get_values()
print(f"Property values computed for {len(values)} annotations")
if values:
    sample = values[0]["values"].get(prop.id, {})
    print(f"Sample metrics: {list(sample.keys())}")

## 6. Connections

Connections link annotations together (e.g., parent-child relationships, lineage tracking).

In [ ]:
# Create two annotations and connect them
parent = ds.annotations.create(ni.Annotation.from_point(
    x=400, y=400, channel=0, tags=["notebook-demo", "parent"], dataset_id=ds.id))
child = ds.annotations.create(ni.Annotation.from_point(
    x=420, y=410, channel=0, tags=["notebook-demo", "child"], dataset_id=ds.id))

conn = ds.connections.create(parent_id=parent.id, child_id=child.id, tags=["lineage"])
print(f"Connection: {conn.id}")
print(f"  parent={conn.parent_id[:12]}  child={conn.child_id[:12]}  tags={conn.tags}")

# List connections
conns = ds.connections.list(parent_id=parent.id)
print(f"Connections from parent: {len(conns)}")

# Auto-connect to nearest
nearby = ds.annotations.create(ni.Annotation.from_point(
    x=405, y=405, channel=0, tags=["notebook-demo", "nearby"], dataset_id=ds.id))
ds.connections.connect_to_nearest(
    annotation_ids=[nearby.id], tags=["parent"], channel=0)
auto_conns = ds.connections.list(node_id=nearby.id)
print(f"Auto-connected: {len(auto_conns)} connections")

## 7. Export

In [ ]:
# JSON export — returns a dict with all data
data = ds.export.to_json()
print(f"JSON export:")
for key, val in data.items():
    count = len(val) if isinstance(val, (list, dict)) else "N/A"
    print(f"  {key}: {count} entries")

# CSV export — returns bytes
csv_bytes = ds.export.to_csv(property_paths=[], delimiter=",")
lines = csv_bytes.decode("utf-8").strip().split("\n")
print(f"\nCSV: {len(lines)} rows")
print(f"Header: {lines[0][:80]}...")

# Save CSV to file
# ds.export.to_csv(property_paths=[], path="export.csv")

## 8. History (undo/redo)

In [ ]:
# History supports undo/redo for annotation operations
count_before = ds.annotations.count()

# Create an annotation
temp = ds.annotations.create(ni.Annotation.from_point(
    x=600, y=600, channel=0, tags=["undo-test"], dataset_id=ds.id))
print(f"After create: {ds.annotations.count()} annotations")

# Undo it
ds.history.undo()
print(f"After undo:   {ds.annotations.count()} annotations")

# Redo it
ds.history.redo()
print(f"After redo:   {ds.annotations.count()} annotations")

# Clean up
ds.annotations.delete_many([a.id for a in ds.annotations.list(tags=["undo-test"])])

## 9. Sharing and access control

In [ ]:
# Check current access
access = ds.sharing.get_access()
print(f"Public: {access.get('public', False)}")
print(f"Users with access: {len(access.get('users', []))}")

# Toggle public access
# ds.sharing.set_public(True)   # make publicly viewable
# ds.sharing.set_public(False)  # restrict access

# Share with another user (by email or username)
# ds.sharing.share("colleague@example.com", access="read")   # read-only
# ds.sharing.share("colleague@example.com", access="write")  # can edit
# ds.sharing.share("colleague@example.com", access="remove") # revoke

## 10. Open in browser

Generate URLs or open datasets directly in the NimbusImage web viewer.

In [ ]:
# Generate URLs without opening
print(f"Info page:   {ds.info_url()}")
print(f"Image viewer: {ds.view_url()}")
print(f"At z=3:      {ds.view_url(z=3)}")

# Open in browser (uncomment to run)
# ds.open()           # opens at default location
# ds.open(z=3)        # opens at z=3
# ds.open(z=3, time=0, layer="multiple")

## Cleanup

Remove all demo annotations and properties created by this notebook.

In [ ]:
# Delete all annotations tagged "notebook-demo"
demo = ds.annotations.list(tags=["notebook-demo"])
if demo:
    # Also delete any connections involving these annotations
    demo_ids = {a.id for a in demo}
    all_conns = ds.connections.list()
    demo_conns = [c for c in all_conns if c.parent_id in demo_ids or c.child_id in demo_ids]
    if demo_conns:
        ds.connections.delete_many([c.id for c in demo_conns])
        print(f"Deleted {len(demo_conns)} connections")

    ds.annotations.delete_many([a.id for a in demo])
    print(f"Deleted {len(demo)} demo annotations")

# Delete demo property and its values
try:
    props = ds.properties.list()
    for p in props:
        if p.name == "Notebook Blob Metrics":
            ds.properties.delete_values(p.id)
            ds.properties.delete(p.id)
            print(f"Deleted property: {p.name}")
except Exception as e:
    print(f"Property cleanup: {e}")

print(f"Final annotation count: {ds.annotations.count()}")